# Notebook 4 — Baseline Evaluation and Comparative Analysis for Llama 3.1 vs Phi‑4

Purpose:
- Load baseline outputs (from Notebook 2)
- Compute deterministic similarity metrics (Rouge L, BLEU, BERTScore, etc.)
- Compute structural completeness as an aggregated score
- Evaluate each section using a Groq‑hosted LLM judge
- Aggregate and normalise metrics
- Generate a radar chart comparing Llama 3.1 and Phi‑4
- (Optional) Log results to MLflow and save reports

In [42]:
from __future__ import annotations

import os
import json
import hashlib
import re
from pydantic import BaseModel
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
import openai

from dotenv import load_dotenv
from openai import OpenAI

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bertscore_score

import plotly.graph_objects as go

In [43]:
# Project paths
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ENV_PATH = PROJECT_ROOT / ".env"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = OUTPUTS_DIR / "reports"
FIGURES_DIR = OUTPUTS_DIR / "figures"
CACHE_DIR = OUTPUTS_DIR / "cache"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ENV_PATH)

# MLflow configuration
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("Environment ready.")
print("MLflow version:", mlflow.__version__)
print("OpenAI version:", openai.__version__)

Environment ready.
MLflow version: 3.10.1
OpenAI version: 2.29.0


## Evaluation settings


In [44]:
CASE_ID = "test_case_004"
FORCE_RERUN_JUDGE = False
RUN_RAGAS = False

SCORE_WEIGHTS = {
    "judge_quality_score": 0.40,
    "semantic_score": 0.20,
    "structure_score": 0.15,
    "safety_score": 0.15,
    "efficiency_score": 0.10,
}

if not np.isclose(sum(SCORE_WEIGHTS.values()), 1.0):
    raise ValueError(f"SCORE_WEIGHTS must sum to 1.0, got {sum(SCORE_WEIGHTS.values()):.3f}")


## Load baseline outputs

In [45]:
OUTPUTS_CSV_PATH = OUTPUTS_DIR / f"baseline_outputs_{CASE_ID}.csv"
if not OUTPUTS_CSV_PATH.exists():
    raise FileNotFoundError(f"Missing outputs file: {OUTPUTS_CSV_PATH}")

outputs_df = pd.read_csv(OUTPUTS_CSV_PATH)

# Confirm that required columns exist
required_cols = [
    "case_id",
    "model_label",
    "provider",
    "latency_sec",
    "prompt_tokens",
    "completion_tokens",
    "total_tokens",
    "output_text",
    "reference_output",
]
missing = [c for c in required_cols if c not in outputs_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Convert latency to milliseconds if desired
outputs_df["latency_ms"] = outputs_df["latency_sec"] * 1000

outputs_df.head()

,case_id,provider,model_label,model_name,prompt_uri,prompt_name,prompt_version,temperature,max_tokens,latency_sec,...,has_balance_sheet_structure_and_liquidity,has_capital_adequacy,has_key_risks_and_watch_points,has_conclusion,structure_score,missing_sections_count,missing_sections,parsed_sections_json,reference_sections_json,latency_ms
0,test_case_004,azure_openai_compatible,Phi-4,Phi-4-experimentation-investment-nextgen,prompts:/financial_analysis_baseline/10,financial_analysis_baseline,10,0.0,1200,28.8203,...,1,1,1,1,1.0,0,[],"{""Executive Summary"": ""Banque Internationale A...","{""Executive Summary"": ""Over the 2021-2024 peri...",28820.3
1,test_case_004,groq,Llama 3.1,llama-3.1-8b-instant,prompts:/financial_analysis_baseline/10,financial_analysis_baseline,10,0.0,1200,8.5730,...,1,1,1,1,1.0,0,[],"{""Executive Summary"": ""Banque Internationale A...","{""Executive Summary"": ""Over the 2021-2024 peri...",8573.0


## Section schema and extraction helpers


In [46]:
EXPECTED_SECTIONS = [
    "Executive Summary",
    "Profitability and Operational Efficiency",
    "Revenue Dynamics",
    "Asset Quality and Risk Profile",
    "Balance Sheet Structure and Liquidity",
    "Capital Adequacy",
    "Key Risks and Watch Points",
    "Conclusion",
]

In [47]:
SECTION_ALIASES = {
    "Executive Summary": ["Executive Summary", "Summary", "Overview"],
    "Profitability and Operational Efficiency": [
        "Profitability and Operational Efficiency",
        "Profitability and Efficiency",
        "Profitability & Efficiency",
    ],
    "Revenue Dynamics": ["Revenue Dynamics"],
    "Asset Quality and Risk Profile": [
        "Asset Quality and Risk Profile",
        "Risk Profile",
    ],
    "Balance Sheet Structure and Liquidity": [
        "Balance Sheet Structure and Liquidity",
        "Liquidity & Balance Sheet",
        "Liquidity and Balance Sheet",
    ],
    "Capital Adequacy": ["Capital Adequacy"],
    "Key Risks and Watch Points": [
        "Key Risks and Watch Points",
        "Risks & Watch Points",
        "Risks and Watch Points",
    ],
    "Conclusion": ["Conclusion"],
}

def normalize_report_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[–—−]", "-", text)
    # Remove bold markdown formatting (**text** -> text)
    text = re.sub(r"\*\*(.+?)\*\*", r"\1", text)
    return text

def extract_sections_rule_based(text: str) -> dict[str, str]:
    text = normalize_report_text(text)
    results = {name: "" for name in EXPECTED_SECTIONS}
    matches = []

    for canonical, aliases in SECTION_ALIASES.items():
        for alias in aliases:
            pattern = rf"(?im)^[ \t]*(?:#+[ \t]*)?{re.escape(alias)}[ \t]*:?[ \t]*$"
            for m in re.finditer(pattern, text):
                matches.append((m.start(), m.end(), canonical, alias))

    matches.sort(key=lambda x: x[0])

    deduped = []
    seen = set()
    for item in matches:
        key = (item[0], item[2])
        if key not in seen:
            deduped.append(item)
            seen.add(key)

    for i, (_, end_pos, canonical, _) in enumerate(deduped):
        next_start = deduped[i + 1][0] if i + 1 < len(deduped) else len(text)
        results[canonical] = text[end_pos:next_start].strip()

    return results

In [48]:
def extract_sections(text: str) -> dict[str, str]:
    return extract_sections_rule_based(text)

## Normalize report text


In [49]:
# Re-normalize outputs with the UPDATED normalize_report_text function
# (This ensures bold markdown is properly removed)
outputs_df["output_text_normalized"] = outputs_df["output_text"].apply(normalize_report_text)
outputs_df["reference_output_normalized"] = outputs_df["reference_output"].apply(normalize_report_text)

print("✅ Re-normalized with updated function that removes bold markdown")

✅ Re-normalized with updated function that removes bold markdown


## Section extraction diagnostics


In [50]:
section_diagnostics = []

for _, row in outputs_df.iterrows():
    for text_col, label in [
        ("reference_output", "reference"),
        ("output_text", "model_output"),
    ]:
        sections = extract_sections(row[text_col])
        present_count = sum(bool(text.strip()) for text in sections.values())
        missing = [name for name, text in sections.items() if not text.strip()]
        section_diagnostics.append({
            "model_label": row["model_label"],
            "text_type": label,
            "sections_found": present_count,
            "sections_expected": len(EXPECTED_SECTIONS),
            "missing_sections": ", ".join(missing),
        })

section_diagnostics_df = pd.DataFrame(section_diagnostics)
display(section_diagnostics_df)


,model_label,text_type,sections_found,sections_expected,missing_sections
0,Phi-4,reference,8,8,
1,Phi-4,model_output,8,8,
2,Llama 3.1,reference,8,8,
3,Llama 3.1,model_output,8,8,


## Metric helpers

In [51]:
# Rouge and BLEU helper objects
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method1

def compute_rouge_l(reference: str, prediction: str) -> float:
    return rouge.score(reference, prediction)["rougeL"].fmeasure

def compute_bleu(reference: str, prediction: str) -> float:
    reference_tokens = reference.split()
    prediction_tokens = prediction.split()
    if not reference_tokens or not prediction_tokens:
        return 0.0
    return sentence_bleu([reference_tokens], prediction_tokens, smoothing_function=smooth)

def compute_output_length(text: str) -> int:
    return len(text.split()) if isinstance(text, str) else 0

def normalized_length_score(prediction: str, reference: str) -> float:
    pred_len = compute_output_length(prediction)
    ref_len = compute_output_length(reference)
    if ref_len == 0:
        return 0.0
    score = 1 - abs(pred_len - ref_len) / ref_len
    return float(max(0.0, min(1.0, score)))



def invert_minmax(series: pd.Series) -> pd.Series:
    series = series.astype(float)
    min_v, max_v = series.min(), series.max()
    if max_v == min_v:
        return pd.Series([1.0] * len(series), index=series.index)
    normalized = (series - min_v) / (max_v - min_v)
    inverted = 1 - normalized
    return inverted.clip(0, 1)

In [52]:
def structure_components(text: str) -> dict:
    sections = extract_sections_rule_based(text)
    present = {k: int(bool(v.strip())) for k, v in sections.items()}
    missing_sections = [k for k, v in sections.items() if not v.strip()]
    structure_score = sum(present.values()) / len(EXPECTED_SECTIONS)

    return {
        "structure_score": structure_score,
        "missing_sections_count": len(missing_sections),
        "missing_sections": ", ".join(missing_sections) if missing_sections else "",
        **{f"has_{k.lower().replace(' ', '_').replace('&', 'and').replace('-', '_')}": v for k, v in present.items()},
    }

## Compute deterministic metrics and structural scores

In [53]:
outputs_df["rougeL"] = outputs_df.apply(
    lambda row: compute_rouge_l(row["reference_output_normalized"], row["output_text_normalized"]),
    axis=1,
)

outputs_df["bleu"] = outputs_df.apply(
    lambda row: compute_bleu(row["reference_output_normalized"], row["output_text_normalized"]),
    axis=1,
)

outputs_df["length_score"] = outputs_df.apply(
    lambda row: normalized_length_score(row["output_text_normalized"], row["reference_output_normalized"]),
    axis=1,
)

structure_columns = [
    "structure_score",
    "missing_sections_count",
    "missing_sections",
] + [col for col in outputs_df.columns if col.startswith("has_")]
outputs_df = outputs_df.drop(columns=[col for col in structure_columns if col in outputs_df.columns])

structure_df = outputs_df["output_text_normalized"].apply(structure_components).apply(pd.Series)
outputs_df = pd.concat([outputs_df, structure_df], axis=1)

P, R, F1 = bertscore_score(
    outputs_df["output_text_normalized"].tolist(),
    outputs_df["reference_output_normalized"].tolist(),
    lang="en",
    verbose=False,
)
outputs_df["bert_precision"] = P.detach().cpu().numpy()
outputs_df["bert_recall"] = R.detach().cpu().numpy()
outputs_df["bert_f1"] = F1.detach().cpu().numpy()

display(outputs_df[["model_label", "rougeL", "bleu", "bert_f1", "length_score", "structure_score"]])

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,model_label,rougeL,bleu,bert_f1,length_score,structure_score
0,Phi-4,0.302932,0.122088,0.891263,0.782344,1.0
1,Llama 3.1,0.288541,0.076551,0.876069,0.757991,1.0


## Judge configuration


In [54]:
# Groq judge settings from .env
JUDGE_API_KEY = os.getenv("JUDGE_API_KEY")      # Groq API key
JUDGE_BASE_URL = os.getenv("JUDGE_BASE_URL")    # e.g. https://api.groq.com/openai/v1
GROQ_JUDGE_MODEL = os.getenv("GROQ_JUDGE_MODEL")  # e.g. "openai/gpt-oss-20b"

# Validate env
required_env = {
    "JUDGE_API_KEY": JUDGE_API_KEY,
    "JUDGE_BASE_URL": JUDGE_BASE_URL,
    "GROQ_JUDGE_MODEL": GROQ_JUDGE_MODEL,
    "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
    "MLFLOW_EXPERIMENT_NAME": MLFLOW_EXPERIMENT_NAME,
}
print("Environment validation:")
for key, val in required_env.items():
    print(f" - {key}: {'OK' if val else 'MISSING'}")
if any(v is None or v == "" for v in required_env.values()):
    raise EnvironmentError("Missing required environment variables.")

Environment validation:
 - JUDGE_API_KEY: OK
 - JUDGE_BASE_URL: OK
 - GROQ_JUDGE_MODEL: OK
 - MLFLOW_TRACKING_URI: OK
 - MLFLOW_EXPERIMENT_NAME: OK


## Judge function


In [55]:
client = OpenAI(
    base_url=JUDGE_BASE_URL,
    api_key=JUDGE_API_KEY,
)
judge_model_name = GROQ_JUDGE_MODEL


def evaluate_with_judge(model_output: str, reference_output: str) -> dict:
    model_sections = extract_sections(model_output)
    ref_sections = extract_sections(reference_output)

    sections_prompt = []
    for name in EXPECTED_SECTIONS:
        model_text = model_sections.get(name, "")
        ref_text = ref_sections.get(name, "")
        sections_prompt.append(
            f"### {name}\n"
            f"Model Output:\n\"\"\"\n{model_text}\n\"\"\"\n"
            f"Ground Truth:\n\"\"\"\n{ref_text}\n\"\"\"\n"
        )
    sections_prompt = "\n".join(sections_prompt)

    json_template = {
        section: {
            "faithfulness": 0,
            "numerical_accuracy": 0,
            "trend_accuracy": 0,
            "reasoning_quality": 0,
            "completeness": 0,
            "hallucination": 0,
            "justification": ""
        }
        for section in EXPECTED_SECTIONS
    }

    system_instruction = (
        "You are a strict evaluation judge for financial reports. "
        "Return ONLY a valid JSON object. "
        "Do not include markdown fences, comments, explanations, or extra text outside JSON. "
        "Each score must be an integer from 0 to 10. "
        "Penalize hallucinations heavily. "
        "If a section is missing or merged, assign a low completeness score and explain why in justification."
    )

    user_prompt = (
        "Evaluate the model output section by section against the ground truth.\n\n"
        "Criteria for each section:\n"
        "- faithfulness\n"
        "- numerical_accuracy\n"
        "- trend_accuracy\n"
        "- reasoning_quality\n"
        "- completeness\n"
        "- hallucination\n\n"
        "Use this exact JSON structure and the same section names:\n"
        f"{json.dumps(json_template, indent=2)}\n\n"
        f"{sections_prompt}"
    )

    response = client.chat.completions.create(
        model=judge_model_name,
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.0,
        max_tokens=3000,
        response_format={"type": "json_object"},
    )

    raw = response.choices[0].message.content.strip()

    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"^```", "", raw)
    raw = re.sub(r"```$", "", raw).strip()

    try:
        parsed = json.loads(raw)
        return parsed
    except Exception:
        return {
            "error": "Failed to parse JSON",
            "raw_response": raw,
        }

## Run the judge


In [56]:
JUDGE_CACHE_PATH = CACHE_DIR / f"judge_eval_{CASE_ID}_{judge_model_name.replace('/', '_')}.json"


def _text_hash(*values: object) -> str:
    payload = "\n".join("" if value is None else str(value) for value in values)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def _judge_cache_key(row: pd.Series) -> str:
    return "|".join([
        str(row["case_id"]),
        str(row["provider"]),
        str(row["model_label"]),
        str(row.get("model_name", "")),
        str(judge_model_name),
        _text_hash(row["output_text"], row["reference_output"]),
    ])


def load_judge_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def save_judge_cache(path: Path, cache: dict) -> None:
    path.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding="utf-8")


judge_cache = {} if FORCE_RERUN_JUDGE else load_judge_cache(JUDGE_CACHE_PATH)
cache_hits = 0
cache_misses = 0
llm_evals = []

for _, row in outputs_df.iterrows():
    cache_key = _judge_cache_key(row)
    if cache_key in judge_cache:
        llm_evals.append(judge_cache[cache_key])
        cache_hits += 1
        continue

    result = evaluate_with_judge(row["output_text"], row["reference_output"])
    judge_cache[cache_key] = result
    llm_evals.append(result)
    cache_misses += 1

outputs_df["llm_eval"] = llm_evals
save_judge_cache(JUDGE_CACHE_PATH, judge_cache)

print(f"Judge cache: {JUDGE_CACHE_PATH}")
print(f"Cache hits: {cache_hits}")
print(f"Cache misses: {cache_misses}")

outputs_df[["model_label", "llm_eval"]]


Judge cache: c:\Users\BRHN\Desktop\SLM-evals\outputs\cache\judge_eval_test_case_004_openai_gpt-oss-20b.json
Cache hits: 2
Cache misses: 0


,model_label,llm_eval
0,Phi-4,"{'Executive Summary': {'faithfulness': 6, 'num..."
1,Llama 3.1,"{'Executive Summary': {'faithfulness': 6, 'num..."


## Flatten judge results


In [57]:
judge_metrics = [
    "faithfulness",
    "numerical_accuracy",
    "trend_accuracy",
    "reasoning_quality",
    "completeness",
    "hallucination",
]

rows = []

for _, row in outputs_df.iterrows():
    eval_dict = row["llm_eval"]
    if not isinstance(eval_dict, dict) or "error" in eval_dict:
        rows.append({
            "model_label": row["model_label"],
            "section": "__evaluation_error__",
            **{m: np.nan for m in judge_metrics},
            "justification": eval_dict.get("raw_response", eval_dict.get("error", "Unknown judge error"))
            if isinstance(eval_dict, dict) else "Invalid judge response",
        })
        continue

    for section in EXPECTED_SECTIONS:
        scores = eval_dict.get(section, {})
        rows.append({
            "model_label": row["model_label"],
            "section": section,
            **{m: scores.get(m) for m in judge_metrics},
            "justification": scores.get("justification", ""),
        })

judge_df = pd.DataFrame(rows)

for metric in judge_metrics:
    judge_df[metric] = pd.to_numeric(judge_df[metric], errors="coerce")

judge_df["quality_score"] = judge_df[
    ["faithfulness", "numerical_accuracy", "trend_accuracy", "reasoning_quality", "completeness"]
].mean(axis=1) / 10
judge_df["safety_score"] = 1 - (judge_df["hallucination"] / 10)
judge_df["section_score"] = (0.80 * judge_df["quality_score"] + 0.20 * judge_df["safety_score"]).clip(0, 1)

display(judge_df)


,model_label,section,faithfulness,numerical_accuracy,trend_accuracy,reasoning_quality,completeness,hallucination,justification,quality_score,safety_score,section_score
0,Phi-4,Executive Summary,6,10,7,6,5,10,"The summary captures growth, profitability, NP...",0.68,0.0,0.544
1,Phi-4,Profitability and Operational Efficiency,8,8,9,8,6,9,The model correctly states the cost‑to‑income ...,0.78,0.1,0.644
2,Phi-4,Revenue Dynamics,9,9,9,8,7,10,"The growth rates for NBI, loans, and deposits ...",0.84,0.0,0.672
3,Phi-4,Asset Quality and Risk Profile,10,10,10,9,9,10,All figures and trends match the ground truth ...,0.96,0.0,0.768
4,Phi-4,Balance Sheet Structure and Liquidity,8,8,8,8,8,6,The model adds specific asset figures and LCR ...,0.80,0.4,0.720
5,Phi-4,Capital Adequacy,10,10,10,9,9,10,The CET1 trend matches the ground truth exactl...,0.96,0.0,0.768
6,Phi-4,Key Risks and Watch Points,5,10,8,7,4,10,The model mentions only a few risks (NBI decel...,0.68,0.0,0.544
7,Phi-4,Conclusion,7,10,8,8,6,10,The conclusion reflects many of the ground tru...,0.78,0.0,0.624
8,Llama 3.1,Executive Summary,6,0,8,5,4,4,Model adds specific growth percentages not pre...,0.46,0.6,0.488
9,Llama 3.1,Profitability and Operational Efficiency,8,10,10,7,6,4,"Model accurately reports net income, ROE, cost...",0.82,0.6,0.776


## Section-level judge insights


In [58]:
section_best_style = "background-color: #fff2cc; color: #7a4f00; font-weight: 700;"

judge_display_columns = {
    "model_label": "Model",
    "section": "Section",
    "section_score": "Section Score",
    "quality_score": "Quality",
    "safety_score": "Safety",
    "faithfulness": "Faithfulness",
    "numerical_accuracy": "Numerical Accuracy",
    "trend_accuracy": "Trend Accuracy",
    "reasoning_quality": "Reasoning",
    "completeness": "Completeness",
    "hallucination": "Hallucination",
    "justification": "Judge Note",
}

judge_display_df = (
    judge_df[judge_df["section"].isin(EXPECTED_SECTIONS)]
    .copy()
    .sort_values(["section", "section_score"], ascending=[True, False])
)
judge_display_df = judge_display_df[list(judge_display_columns)].rename(columns=judge_display_columns)

section_metric_directions = {
    "Section Score": "max",
    "Quality": "max",
    "Safety": "max",
    "Faithfulness": "max",
    "Numerical Accuracy": "max",
    "Trend Accuracy": "max",
    "Reasoning": "max",
    "Completeness": "max",
    "Hallucination": "min",
}

def highlight_section_best_values(column: pd.Series) -> list[str]:
    direction = section_metric_directions.get(column.name)
    if direction is None:
        return [""] * len(column)

    values = pd.to_numeric(column, errors="coerce")
    if values.notna().sum() == 0:
        return [""] * len(column)

    best_value = values.min() if direction == "min" else values.max()
    return [section_best_style if pd.notna(value) and value == best_value else "" for value in values]

judge_percent_columns = ["Section Score", "Quality", "Safety"]
judge_raw_score_columns = [
    "Faithfulness",
    "Numerical Accuracy",
    "Trend Accuracy",
    "Reasoning",
    "Completeness",
    "Hallucination",
]

judge_styler = (
    judge_display_df.style
    .format({col: "{:.1%}" for col in judge_percent_columns})
    .format({col: "{:.1f}" for col in judge_raw_score_columns})
    .apply(highlight_section_best_values, axis=0)
    .set_properties(subset=["Judge Note"], **{"max-width": "520px", "white-space": "normal"})
    .set_caption("Section-Level Judge Scores - best values highlighted in gold")
)

display(judge_styler)

section_winners_df = (
    judge_df[judge_df["section"].isin(EXPECTED_SECTIONS)]
    .sort_values(["section", "section_score"], ascending=[True, False])
    .groupby("section", as_index=False)
    .first()
)

print("Section winners")
for _, row in section_winners_df.iterrows():
    print(f"- {row['section']}: {row['model_label']} ({row['section_score']:.1%}).")

weakest_section_rows = []
for model_label, model_sections in judge_df[judge_df["section"].isin(EXPECTED_SECTIONS)].groupby("model_label"):
    weakest = model_sections.sort_values("section_score", ascending=True).head(2)
    for position, (_, row) in enumerate(weakest.iterrows(), start=1):
        weakest_section_rows.append({
            "model_label": model_label,
            "weakness_rank": position,
            "section": row["section"],
            "section_score": row["section_score"],
            "quality_score": row["quality_score"],
            "safety_score": row["safety_score"],
            "justification": row["justification"],
        })

weakest_sections_df = pd.DataFrame(weakest_section_rows)

print()
print("Weakest sections by model")
for model_label, weakest in weakest_sections_df.groupby("model_label"):
    weakness_text = "; ".join(
        f"{row['section']} ({row['section_score']:.1%})"
        for _, row in weakest.iterrows()
    )
    print(f"- {model_label}: {weakness_text}.")


,Model,Section,Section Score,Quality,Safety,Faithfulness,Numerical Accuracy,Trend Accuracy,Reasoning,Completeness,Hallucination,Judge Note
3,Phi-4,Asset Quality and Risk Profile,0.768000,0.960000,0.000000,10.0,10.0,10.0,9.0,9.0,10.0,"All figures and trends match the ground truth exactly. The reasoning is concise but accurate, with no missing or fabricated information."
11,Llama 3.1,Asset Quality and Risk Profile,0.636000,0.620000,0.700000,7.0,8.0,7.0,5.0,4.0,3.0,Model matches NPL and coverage ratio; cost of risk trend mismatch; small hallucination.
4,Phi-4,Balance Sheet Structure and Liquidity,0.720000,0.800000,0.400000,8.0,8.0,8.0,8.0,8.0,6.0,"The model adds specific asset figures and LCR values not present in the ground truth, which may be plausible but are not verified. The overall trends and ratios are consistent with the ground truth. Minor hallucination risk due to unverified numbers."
12,Llama 3.1,Balance Sheet Structure and Liquidity,0.552000,0.540000,0.600000,6.0,4.0,8.0,5.0,4.0,4.0,Model states asset growth 25% not in ground truth; loan-to-deposit ratio correct; missing liquidity metrics; hallucination.
13,Llama 3.1,Capital Adequacy,0.872000,0.840000,1.000000,10.0,10.0,10.0,5.0,7.0,0.0,Model fully matches ground truth; no hallucination.
5,Phi-4,Capital Adequacy,0.768000,0.960000,0.000000,10.0,10.0,10.0,9.0,9.0,10.0,The CET1 trend matches the ground truth exactly. No hallucinations or omissions.
7,Phi-4,Conclusion,0.624000,0.780000,0.000000,7.0,10.0,8.0,8.0,6.0,10.0,"The conclusion reflects many of the ground truth points (profitability, efficiency, asset quality, capital, liquidity) but misses macro‑environment concerns, controlled credit risk, and the moderate growth outlook. No hallucinations are present."
15,Llama 3.1,Conclusion,0.408000,0.260000,1.000000,5.0,0.0,0.0,5.0,3.0,0.0,Model repeats some points but omits many; incomplete; no hallucination.
0,Phi-4,Executive Summary,0.544000,0.680000,0.000000,6.0,10.0,7.0,6.0,5.0,10.0,"The summary captures growth, profitability, NPL decline, and CET1 rise, but omits the transition to a mature model, controlled risk metrics, liquidity buffers, and macro‑environment context. No numerical data are provided, so numerical accuracy is high. Trend accuracy is moderate because the model does not mention the moderation of growth. Reasoning is basic and incomplete. No hallucinated facts are present."
8,Llama 3.1,Executive Summary,0.488000,0.460000,0.600000,6.0,0.0,8.0,5.0,4.0,4.0,"Model adds specific growth percentages not present in ground truth, but correctly states overall improvement. Numbers are hallucinated, so faithfulness moderate, hallucination present."


Section winners
- Asset Quality and Risk Profile: Phi-4 (76.8%).
- Balance Sheet Structure and Liquidity: Phi-4 (72.0%).
- Capital Adequacy: Llama 3.1 (87.2%).
- Conclusion: Phi-4 (62.4%).
- Executive Summary: Phi-4 (54.4%).
- Key Risks and Watch Points: Phi-4 (54.4%).
- Profitability and Operational Efficiency: Llama 3.1 (77.6%).
- Revenue Dynamics: Phi-4 (67.2%).

Weakest sections by model
- Llama 3.1: Key Risks and Watch Points (36.0%); Conclusion (40.8%).
- Phi-4: Executive Summary (54.4%); Key Risks and Watch Points (54.4%).


## Aggregate judge scores by model


In [59]:
aggregated_judge_scores = (
    judge_df.groupby("model_label")[judge_metrics]
    .mean()
    .reset_index()
)

display(aggregated_judge_scores)

,model_label,faithfulness,numerical_accuracy,trend_accuracy,reasoning_quality,completeness,hallucination
0,Llama 3.1,6.500,4.875,6.125,5.125,4.25,2.375
1,Phi-4,7.875,9.375,8.625,7.875,6.75,9.375


## Score model
The final score is a weighted 0-1 score designed for model comparison. Judge quality carries the most weight because factuality, numerical accuracy, trend accuracy, reasoning, and completeness matter more than lexical overlap for financial reporting. Semantic similarity captures reference alignment, structure rewards complete section coverage, safety rewards low hallucination, and efficiency rewards lower latency.


## Build comparison table


In [60]:
comparison_df = outputs_df[
    [
        "model_label",
        "provider",
        "latency_ms",
        "rougeL",
        "bleu",
        "bert_f1",
        "length_score",
        "structure_score",
    ]
].copy()

comparison_df = comparison_df.merge(aggregated_judge_scores, on="model_label")
comparison_df["latency_score"] = invert_minmax(comparison_df["latency_ms"])

# Keep raw judge averages in 0-10 form, and add normalized 0-1 score columns for charts/scoring.
for metric in judge_metrics:
    comparison_df[f"{metric}_score"] = (
        pd.to_numeric(comparison_df[metric], errors="coerce") / 10
    ).fillna(0.0).clip(0, 1)

# Higher hallucination is worse, so invert it for any "higher is better" visualization.
comparison_df["non_hallucination_score"] = 1 - comparison_df["hallucination_score"]

semantic_metrics = ["rougeL", "bert_f1"]
judge_quality_metrics = [
    "faithfulness_score",
    "numerical_accuracy_score",
    "trend_accuracy_score",
    "reasoning_quality_score",
    "completeness_score",
]

comparison_df["semantic_score"] = comparison_df[semantic_metrics].mean(axis=1)
comparison_df["judge_quality_score"] = comparison_df[judge_quality_metrics].mean(axis=1)
comparison_df["safety_score"] = comparison_df["non_hallucination_score"]
comparison_df["efficiency_score"] = comparison_df["latency_score"]

comparison_df["final_score"] = sum(
    comparison_df[metric] * weight
    for metric, weight in SCORE_WEIGHTS.items()
)
comparison_df["rank"] = comparison_df["final_score"].rank(ascending=False, method="dense").astype(int)

comparison_display_columns = {
    "rank": "Rank",
    "model_label": "Model",
    "provider": "Provider",
    "final_score": "Final Score",
    "judge_quality_score": "Judge Quality",
    "semantic_score": "Semantic Similarity",
    "structure_score": "Structure",
    "safety_score": "Safety",
    "efficiency_score": "Efficiency",
    "latency_ms": "Latency (ms)",
    "rougeL": "Rouge-L",
    "bert_f1": "BERTScore F1",
    "bleu": "BLEU",
}

comparison_display_df = (
    comparison_df[list(comparison_display_columns)]
    .rename(columns=comparison_display_columns)
    .sort_values(["Rank", "Model"])
    .reset_index(drop=True)
)

metric_directions = {
    "Rank": "min",
    "Latency (ms)": "min",
    "Final Score": "max",
    "Judge Quality": "max",
    "Semantic Similarity": "max",
    "Structure": "max",
    "Safety": "max",
    "Efficiency": "max",
    "Rouge-L": "max",
    "BERTScore F1": "max",
    "BLEU": "max",
}

best_style = "background-color: #fff2cc; color: #7a4f00; font-weight: 700;"

def highlight_best_values(column: pd.Series) -> list[str]:
    direction = metric_directions.get(column.name)
    if direction is None:
        return [""] * len(column)

    values = pd.to_numeric(column, errors="coerce")
    if values.notna().sum() == 0:
        return [""] * len(column)

    best_value = values.min() if direction == "min" else values.max()
    return [best_style if pd.notna(value) and value == best_value else "" for value in values]

percent_columns = [
    "Final Score",
    "Judge Quality",
    "Semantic Similarity",
    "Structure",
    "Safety",
    "Efficiency",
    "Rouge-L",
    "BERTScore F1",
    "BLEU",
]

comparison_styler = (
    comparison_display_df.style
    .format({col: "{:.1%}" for col in percent_columns})
    .format({"Latency (ms)": "{:.0f}", "Rank": "{:.0f}"})
    .apply(highlight_best_values, axis=0)
    .set_caption("Model Comparison Summary - best values highlighted in gold")
)

display(comparison_styler)

winner = comparison_df.sort_values("final_score", ascending=False).iloc[0]
fastest = comparison_df.sort_values("latency_ms", ascending=True).iloc[0]
most_accurate = comparison_df.sort_values("numerical_accuracy_score", ascending=False).iloc[0]

print("Evaluation highlights")
print(f"- Best overall: {winner['model_label']} with final score {winner['final_score']:.1%}.")
print(f"- Fastest model: {fastest['model_label']} at {fastest['latency_ms']:.0f} ms.")
print(f"- Best numerical accuracy: {most_accurate['model_label']} at {most_accurate['numerical_accuracy_score']:.1%}.")

score_columns = [
    "judge_quality_score",
    "semantic_score",
    "structure_score",
    "safety_score",
    "efficiency_score",
]
score_labels = {
    "judge_quality_score": "judge quality",
    "semantic_score": "semantic similarity",
    "structure_score": "structure",
    "safety_score": "safety",
    "efficiency_score": "efficiency",
}

for _, row in comparison_df.sort_values("rank").iterrows():
    weakest_metric = min(score_columns, key=lambda metric: row[metric])
    print(
        f"- Main weakness for {row['model_label']}: "
        f"{score_labels[weakest_metric]} ({row[weakest_metric]:.1%})."
    )


,Rank,Model,Provider,Final Score,Judge Quality,Semantic Similarity,Structure,Safety,Efficiency,Latency (ms),Rouge-L,BERTScore F1,BLEU
0,1,Llama 3.1,groq,0.695836,0.537500,0.582305,1.000000,0.762500,1.000000,8573,0.288541,0.876069,0.076551
1,2,Phi-4,azure_openai_compatible,0.602794,0.810000,0.597097,1.000000,0.062500,0.000000,28820,0.302932,0.891263,0.122088


Evaluation highlights
- Best overall: Llama 3.1 with final score 69.6%.
- Fastest model: Llama 3.1 at 8573 ms.
- Best numerical accuracy: Phi-4 at 93.8%.
- Main weakness for Llama 3.1: judge quality (53.8%).
- Main weakness for Phi-4: efficiency (0.0%).


## Build radar chart


In [61]:
def plot_plotly_radar(df: pd.DataFrame, model_col: str, metrics: list[str], title: str):
    categories = metrics + [metrics[0]]

    fig = go.Figure()

    for _, row in df.iterrows():
        values = row[metrics].tolist()
        values += values[:1]

        fig.add_trace(
            go.Scatterpolar(
                r=values,
                theta=categories,
                fill="toself",
                name=row[model_col],
            )
        )

    fig.update_layout(
        title=title,
        polar=dict(
            radialaxis=dict(visible=True, range=[0, 1]),
        ),
        showlegend=True,
    )
    return fig


judge_radar_metrics = [
    "faithfulness_score",
    "numerical_accuracy_score",
    "trend_accuracy_score",
    "reasoning_quality_score",
    "completeness_score",
    "non_hallucination_score",
]

radar_metrics = [
    "rougeL",
    "bleu",
    "bert_f1",
    "length_score",
    "structure_score",
    "latency_score",
] + judge_radar_metrics

for col in radar_metrics:
    comparison_df[col] = pd.to_numeric(comparison_df[col], errors="coerce").fillna(0.0).clip(0, 1)

radar_fig = plot_plotly_radar(
    comparison_df,
    model_col="model_label",
    metrics=radar_metrics,
    title="Evaluation Radar Chart: Llama 3.1 vs Phi-4",
)

radar_fig.show()

radar_html_path = FIGURES_DIR / f"radar_comparison_{CASE_ID}.html"
radar_png_path = FIGURES_DIR / f"radar_comparison_{CASE_ID}.png"

radar_fig.write_html(str(radar_html_path))
radar_fig.write_image(str(radar_png_path))

print("Saved:", radar_html_path)
print("Saved:", radar_png_path)

Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\figures\radar_comparison_test_case_004.html
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\figures\radar_comparison_test_case_004.png


## Save reports


In [62]:
comparison_csv_path = REPORTS_DIR / f"comparison_metrics_{CASE_ID}_nb4.csv"
comparison_json_path = REPORTS_DIR / f"comparison_metrics_{CASE_ID}_nb4.json"
comparison_styled_html_path = REPORTS_DIR / f"comparison_metrics_{CASE_ID}_nb4_styled.html"
comparison_summary_csv_path = REPORTS_DIR / f"comparison_summary_{CASE_ID}_nb4.csv"
judge_csv_path = REPORTS_DIR / f"judge_section_scores_{CASE_ID}_nb4.csv"
judge_styled_html_path = REPORTS_DIR / f"judge_section_scores_{CASE_ID}_nb4_styled.html"
section_winners_csv_path = REPORTS_DIR / f"section_winners_{CASE_ID}_nb4.csv"
weakest_sections_csv_path = REPORTS_DIR / f"weakest_sections_{CASE_ID}_nb4.csv"

comparison_df.to_csv(comparison_csv_path, index=False, encoding="utf-8")
comparison_df.to_json(comparison_json_path, orient="records", force_ascii=False, indent=2)
comparison_styled_html_path.write_text(comparison_styler.to_html(), encoding="utf-8")
comparison_display_df.to_csv(comparison_summary_csv_path, index=False, encoding="utf-8")
judge_df.to_csv(judge_csv_path, index=False, encoding="utf-8")
judge_styled_html_path.write_text(judge_styler.to_html(), encoding="utf-8")
section_winners_df.to_csv(section_winners_csv_path, index=False, encoding="utf-8")
weakest_sections_df.to_csv(weakest_sections_csv_path, index=False, encoding="utf-8")

print("Saved:", comparison_csv_path)
print("Saved:", comparison_json_path)
print("Saved:", comparison_styled_html_path)
print("Saved:", comparison_summary_csv_path)
print("Saved:", judge_csv_path)
print("Saved:", judge_styled_html_path)
print("Saved:", section_winners_csv_path)
print("Saved:", weakest_sections_csv_path)


Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\comparison_metrics_test_case_004_nb4.csv
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\comparison_metrics_test_case_004_nb4.json
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\comparison_metrics_test_case_004_nb4_styled.html
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\comparison_summary_test_case_004_nb4.csv
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\judge_section_scores_test_case_004_nb4.csv
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\judge_section_scores_test_case_004_nb4_styled.html
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\section_winners_test_case_004_nb4.csv
Saved: c:\Users\BRHN\Desktop\SLM-evals\outputs\reports\weakest_sections_test_case_004_nb4.csv


## MLflow logging and reports

In [63]:
with mlflow.start_run(run_name=f"nb4_evaluation_{CASE_ID}") as run:
    eval_run_id = run.info.run_id

    mlflow.log_params({
        "phase": "evaluation",
        "case_id": CASE_ID,
        "rows_evaluated": len(outputs_df),
        "judge_model": judge_model_name,
        "source_outputs_csv": str(OUTPUTS_CSV_PATH),
        "judge_cache_path": str(JUDGE_CACHE_PATH),
        "force_rerun_judge": FORCE_RERUN_JUDGE,
        "run_ragas": RUN_RAGAS,
        "score_weights": json.dumps(SCORE_WEIGHTS),
    })

    mlflow.log_metrics({
        "best_final_score": float(comparison_df["final_score"].max()),
        "mean_final_score": float(comparison_df["final_score"].mean()),
        "mean_judge_quality_score": float(comparison_df["judge_quality_score"].mean()),
        "mean_safety_score": float(comparison_df["safety_score"].mean()),
        "mean_section_score": float(judge_df["section_score"].mean()),
        "judge_cache_hits": float(cache_hits),
        "judge_cache_misses": float(cache_misses),
    })

    mlflow.log_artifact(str(comparison_csv_path), artifact_path="evaluation_reports")
    mlflow.log_artifact(str(comparison_json_path), artifact_path="evaluation_reports")
    mlflow.log_artifact(str(comparison_styled_html_path), artifact_path="evaluation_reports")
    mlflow.log_artifact(str(comparison_summary_csv_path), artifact_path="evaluation_reports")
    mlflow.log_artifact(str(judge_csv_path), artifact_path="evaluation_reports")
    mlflow.log_artifact(str(judge_styled_html_path), artifact_path="evaluation_reports")
    mlflow.log_artifact(str(section_winners_csv_path), artifact_path="evaluation_reports")
    mlflow.log_artifact(str(weakest_sections_csv_path), artifact_path="evaluation_reports")
    mlflow.log_artifact(str(JUDGE_CACHE_PATH), artifact_path="evaluation_cache")
    mlflow.log_artifact(str(radar_html_path), artifact_path="evaluation_figures")
    mlflow.log_artifact(str(radar_png_path), artifact_path="evaluation_figures")

print("MLflow logging done.")


🏃 View run nb4_evaluation_test_case_004 at: http://127.0.0.1:5000/#/experiments/4/runs/271babfbb1ae426a874a28ee9dcc2a9c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
MLflow logging done.


## Optional RAGAS metrics


In [64]:
ragas_scores = None

if RUN_RAGAS:
    from ragas import evaluate
    from ragas.metrics import answer_correctness, answer_relevancy
    from datasets import Dataset

    records = []
    for _, row in outputs_df.iterrows():
        for section in EXPECTED_SECTIONS:
            pred = extract_sections(row["output_text"])[section]
            ref = extract_sections(row["reference_output"])[section]
            if pred and ref:
                records.append({
                    "query": f"{row['case_id']} - {section}",
                    "generated_response": pred,
                    "answer": ref,
                })

    if not records:
        print("RAGAS skipped: no valid section records were available.")
    else:
        dataset = Dataset.from_list(records)
        ragas_scores = evaluate(dataset, [answer_correctness, answer_relevancy])
        print("RAGAS metrics:", ragas_scores)
else:
    print("RAGAS skipped. Set RUN_RAGAS = True in Evaluation settings to enable it.")


RAGAS skipped. Set RUN_RAGAS = True in Evaluation settings to enable it.
